# DL · Encuentro 1 · Neurona, activaciones y entrenamiento

Cómo aprende una red neuronal

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Una red transforma entradas](#scrollTo=tema-dl-redes)
2. [La activación cambia la respuesta](#scrollTo=tema-dl-activaciones)
3. [El error guía los ajustes](#scrollTo=tema-dl-entrenamiento)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · 01 · Una cuenta y 09 · Una neurona

**Lo que vas a poder hacer:** Construir el cálculo de una neurona y rastrear cómo sus entradas producen una salida.

Permite explicar el cálculo antes de confiar en una etiqueta o una puntuación.

### Entiende la idea

Una red neuronal es un conjunto de pequeñas unidades de cálculo conectadas. No es un cerebro ni entiende por ver una imagen. Recibe números y los transforma. Una neurona es una de esas unidades: multiplica cada entrada por un valor llamado peso, suma los resultados y añade otro número llamado sesgo. Después aplica una regla, la activación, para obtener su salida. Las unidades pueden organizarse en capas: la salida de una alimenta a la siguiente. Primero seguiremos una sola neurona con valores dados. Aprender será cambiar pesos y sesgos a partir de ejemplos; esta cuenta aún no los cambia.

### Un caso pequeño antes del programa

**Dos señales y un ajuste**

Desarmamos una unidad de cálculo que podría formar parte de una red para imágenes. Sus entradas x₁ = 2 y x₂ = 1 son dos señales numéricas abstractas, no cantidades de frutos ni porcentajes de enfermedad. Los pesos w₁ = 0,5 y w₂ = −1 y el sesgo b = 0,5 se fijan para seguir la operación. Aplicamos ReLU, que conserva la suma si es positiva y da 0 si es negativa.

| Parte | Entrada | Peso | Contribución |
| --- | --- | --- | --- |
| Señal 1 | 2 | 0,5 | 1 |
| Señal 2 | 1 | −1 | −1 |
| Sesgo añadido | — | — | 0,5 |

1. Multiplica las entradas: 2 × 0,5 = 1 y 1 × (−1) = −1.
2. Suma los aportes y el sesgo: z = 1 − 1 + 0,5 = 0,5.
3. ReLU conserva las sumas positivas: la salida es 0,5. Si el sesgo pasa a −0,5, z = −0,5 y la salida es 0.

**Qué demuestra:** Una sola entrada o un sesgo pueden cambiar la salida sin que haya ocurrido aprendizaje.

### Ahora identifica los datos de esta práctica

Reconstruye entradas, productos, sesgo y salida. En el cuaderno se muestra sigmoide: no esperes el mismo número que en una salida ReLU.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Cambiar una entrada y recalcular usa los pesos existentes. Eso es inferencia; todavía no significa que la neurona haya aprendido.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL1: nombres que encontrarás al calcular y entrenar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DL** — DL: Deep Learning (Aprendizaje profundo). Parte del aprendizaje automático que usa redes neuronales con varias capas para transformar información y aprender representaciones. Una capa transforma números y entrega otros a la siguiente. Más capas no garantizan mejores resultados. [Fuente](https://cloud.google.com/discover/what-is-deep-learning)

- **ReLU** — ReLU: Rectified Linear Unit (Unidad lineal rectificada). Función de activación que devuelve el mayor entre la entrada y cero. Conserva números positivos y convierte los negativos en cero. Su salida no tiene límite superior y no representa por sí sola una probabilidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **BCE** — BCE: Binary Cross Entropy (Entropía cruzada binaria). Pérdida que compara una salida binaria probabilística con su respuesta conocida. En el caso sin pesos calcula −[y ln(p)+(1−y) ln(1−p)]. No tiene unidades físicas y no equivale a un porcentaje de errores; cuanto menor, mejor ajuste de esa pérdida sobre los mismos casos. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

- **BCEWithLogitsLoss** — BCEWithLogitsLoss: Binary Cross Entropy with Logits Loss (Pérdida de entropía cruzada binaria a partir de logits). Función de PyTorch que recibe logits y combina internamente sigmoide con entropía cruzada binaria. Evita separar ambas operaciones para mejorar la estabilidad numérica. En el cuaderno no se añade una sigmoide antes de pasar la salida a esta pérdida. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

- **Adam** — Adam: Adaptive Moment Estimation (Estimación adaptativa de momentos). Optimizador que ajusta los parámetros usando promedios de los gradientes y de sus cuadrados. El nombre procede de esas estimaciones adaptativas. Tiene una tasa global y un ajuste dependiente del historial para cada parámetro; no aplica simplemente el mismo paso del descenso básico. [Fuente](https://arxiv.org/abs/1412.6980)


In [ ]:
entradas = np.array([.6,.2]); pesos = np.array([2.,-1.]); sesgo = -.5
contribuciones = entradas * pesos
z = entradas @ pesos + sesgo
print('Contribuciones:',contribuciones,'· sesgo:',sesgo,'· suma z:',z)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(['Color × peso','Manchas × peso','Sesgo'],[*contribuciones,sesgo],color=[AZUL,VERDE,GRIS])
ax.axhline(0,color='black',lw=.7)
ax.set(title=f'La suma de estas contribuciones es z = {z:.1f}',ylabel='Aporte a la suma interna')
mostrar('01-neurona')


### Después de ejecutar: sigue las contribuciones

Comprueba 1,2 − 0,2 − 0,5 = 0,5. No confundas la barra negativa con un número negativo de manchas: es la contribución del peso elegido. La figura muestra cómo se forma una salida; no mide todavía si esa salida acierta.


### 01B · Corregir un ajuste paso a paso

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Una neurona con parámetros explícitos. No está entrenada: reproduce la guía.
capa=nn.Linear(2,1)
with torch.no_grad():
    capa.weight.copy_(torch.tensor([[2.,-1.]]));capa.bias.copy_(torch.tensor([-.5]))
x=torch.tensor([[.6,.2]])
z=capa(x);p=torch.sigmoid(z)
print('Entradas:',x.tolist(),'· pesos:',capa.weight.detach().tolist(),'· sesgo:',capa.bias.item())
print('Puntuación z:',z.item(),'· salida sigmoid del ejemplo:',p.item())
print('Los parámetros se fijaron a mano: esta salida no es una probabilidad validada de que un grano esté sano.')
valores=torch.linspace(-4,4,150)
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(nombre,f) in zip(axs,[('Identidad',lambda x:x),('Sigmoide',torch.sigmoid),('tanh',torch.tanh),('ReLU',torch.relu)]):
    ax.plot(valores,f(valores),color=AZUL);ax.axhline(0,color=GRIS,lw=.7);ax.set(title=nombre,xlabel='z',ylabel='f(z)')
mostrar('09-activaciones')
print('Softmax [2,1,0]:',torch.softmax(torch.tensor([2.,1.,0.]),dim=0).tolist())


### Después de ejecutar: suma y transformación son pasos distintos

La suma z es 0,5 y su sigmoide es aproximadamente 0,622. ReLU sobre esa misma suma daría 0,5; no deben salir iguales por ser la misma neurona. Los pesos están dados; ninguna de estas cifras prueba que un grano real sea sano.


### 09B · Entrenar una red pequeña

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


### Lee, comprueba y explica

Reconstruye entradas, productos, sesgo y salida. En el cuaderno se muestra sigmoide: no esperes el mismo número que en una salida ReLU.

**Prueba pequeña:** Cambiar una entrada y recalcular usa los pesos existentes. Eso es inferencia; todavía no significa que la neurona haya aprendido.

**Vuelve al caso:** Sigue entradas → productos por pesos → suma con sesgo → activación. Ese recorrido permite comprobar una neurona antes de interpretar una red.

**Pregunta:** ¿Qué diferencia una neurona, una capa y una red?

**Respuesta razonada:** Una neurona calcula una transformación; una capa reúne transformaciones que producen una representación; una red conecta capas. El entrenamiento cambia pesos y sesgos, mientras la inferencia usa los valores ya fijados.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-redes)


## Tema 2 · 09 · Curvas de activación

**Lo que vas a poder hacer:** Configurar una transformación de señales y comprobar su rango de salida.

Ayuda a evitar que un cero, un negativo o un valor entre cero y uno se confundan con una categoría o un acierto.

### Entiende la idea

Una activación es una regla que recibe la suma de una neurona y la convierte en el número que seguirá adelante. No añade una foto ni decide por sí sola si hay un defecto. ReLU devuelve cero si recibe un número negativo y conserva los positivos. La sigmoide lleva cualquier entrada finita a un valor entre cero y uno. Tanh la lleva entre menos uno y uno. Estas transformaciones permiten combinar señales de manera más flexible que repetir solo sumas y multiplicaciones. Veremos la misma entrada en las tres, antes de leer sus curvas.

### Un caso pequeño antes del programa

**La misma señal sigue tres reglas**

Aplicamos ReLU, sigmoid y tanh a cuatro números de entrada. Cada fila conserva la misma entrada para comparar las tres funciones; las columnas contienen sus salidas, redondeadas a tres decimales.

| Entrada | ReLU | Sigmoid, aprox. | tanh, aprox. |
| --- | --- | --- | --- |
| −2 | 0 | 0,119 | −0,964 |
| 0 | 0 | 0,500 | 0 |
| 0,5 | 0,5 | 0,622 | 0,462 |
| 2 | 2 | 0,881 | 0,964 |

1. En la fila −2, ReLU compara con cero y conserva el mayor: devuelve 0.
2. Sigmoid comprime esa entrada a 0,119; tanh conserva el signo y devuelve aproximadamente −0,964.
3. En la fila 2, ReLU devuelve 2. Una salida superior a 1 confirma que no todas las activaciones expresan probabilidades.

**Qué demuestra:** Cambiar la función cambia el rango y la respuesta, aunque la entrada sea idéntica.

### Ahora identifica los datos de esta práctica

Elige la misma entrada en las curvas y compara qué devuelve cada función. Horizontal: entrada; vertical: salida transformada.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Con entrada 0, ReLU y tanh dan 0 y sigmoide da 0,5. Esa diferencia pertenece a la función, no a otra imagen ni a un nuevo entrenamiento.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL1: nombres que encontrarás al calcular y entrenar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DL** — DL: Deep Learning (Aprendizaje profundo). Parte del aprendizaje automático que usa redes neuronales con varias capas para transformar información y aprender representaciones. Una capa transforma números y entrega otros a la siguiente. Más capas no garantizan mejores resultados. [Fuente](https://cloud.google.com/discover/what-is-deep-learning)

- **ReLU** — ReLU: Rectified Linear Unit (Unidad lineal rectificada). Función de activación que devuelve el mayor entre la entrada y cero. Conserva números positivos y convierte los negativos en cero. Su salida no tiene límite superior y no representa por sí sola una probabilidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **BCE** — BCE: Binary Cross Entropy (Entropía cruzada binaria). Pérdida que compara una salida binaria probabilística con su respuesta conocida. En el caso sin pesos calcula −[y ln(p)+(1−y) ln(1−p)]. No tiene unidades físicas y no equivale a un porcentaje de errores; cuanto menor, mejor ajuste de esa pérdida sobre los mismos casos. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

- **BCEWithLogitsLoss** — BCEWithLogitsLoss: Binary Cross Entropy with Logits Loss (Pérdida de entropía cruzada binaria a partir de logits). Función de PyTorch que recibe logits y combina internamente sigmoide con entropía cruzada binaria. Evita separar ambas operaciones para mejorar la estabilidad numérica. En el cuaderno no se añade una sigmoide antes de pasar la salida a esta pérdida. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

- **Adam** — Adam: Adaptive Moment Estimation (Estimación adaptativa de momentos). Optimizador que ajusta los parámetros usando promedios de los gradientes y de sus cuadrados. El nombre procede de esas estimaciones adaptativas. Tiene una tasa global y un ajuste dependiente del historial para cada parámetro; no aplica simplemente el mismo paso del descenso básico. [Fuente](https://arxiv.org/abs/1412.6980)


In [ ]:
# Una neurona con parámetros explícitos. No está entrenada: reproduce la guía.
capa=nn.Linear(2,1)
with torch.no_grad():
    capa.weight.copy_(torch.tensor([[2.,-1.]]));capa.bias.copy_(torch.tensor([-.5]))
x=torch.tensor([[.6,.2]])
z=capa(x);p=torch.sigmoid(z)
print('Entradas:',x.tolist(),'· pesos:',capa.weight.detach().tolist(),'· sesgo:',capa.bias.item())
print('Puntuación z:',z.item(),'· salida sigmoid del ejemplo:',p.item())
print('Los parámetros se fijaron a mano: esta salida no es una probabilidad validada de que un grano esté sano.')
valores=torch.linspace(-4,4,150)
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(nombre,f) in zip(axs,[('Identidad',lambda x:x),('Sigmoide',torch.sigmoid),('tanh',torch.tanh),('ReLU',torch.relu)]):
    ax.plot(valores,f(valores),color=AZUL);ax.axhline(0,color=GRIS,lw=.7);ax.set(title=nombre,xlabel='z',ylabel='f(z)')
mostrar('09-activaciones')
print('Softmax [2,1,0]:',torch.softmax(torch.tensor([2.,1.,0.]),dim=0).tolist())


### Después de ejecutar: suma y transformación son pasos distintos

La suma z es 0,5 y su sigmoide es aproximadamente 0,622. ReLU sobre esa misma suma daría 0,5; no deben salir iguales por ser la misma neurona. Los pesos están dados; ninguna de estas cifras prueba que un grano real sea sano.


### 09B · Entrenar una red pequeña

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Aplicamos la misma señal a todas las funciones, sin entrenar una red.
SENAL = -2.0  # Cambia solo esta entrada y vuelve a ejecutar esta celda.
senal = torch.tensor(SENAL)
comparacion_activaciones = pd.DataFrame({
    'Función': ['Identidad', 'Sigmoide', 'tanh', 'ReLU'],
    'Entrada': [SENAL]*4,
    'Salida': [SENAL, torch.sigmoid(senal).item(), torch.tanh(senal).item(), torch.relu(senal).item()]})
display(comparacion_activaciones)
print('La entrada es una señal numérica interna, no una cantidad negativa de productos.')
print('Cambiar la señal utiliza las funciones existentes; no ajusta pesos ni entrena un modelo.')


### Lee, comprueba y explica

Elige la misma entrada en las curvas y compara qué devuelve cada función. Horizontal: entrada; vertical: salida transformada.

**Prueba pequeña:** Con entrada 0, ReLU y tanh dan 0 y sigmoide da 0,5. Esa diferencia pertenece a la función, no a otra imagen ni a un nuevo entrenamiento.

**Vuelve al caso:** Compara activaciones manteniendo la entrada fija. Verifica su rango antes de interpretar la salida como puntuación o conectarla a otra operación.

**Pregunta:** ¿Por qué apilar capas sin activaciones no lineales puede aportar menos de lo esperado?

**Respuesta razonada:** Si cada capa solo multiplica y suma, podemos juntar todas esas cuentas en una sola regla del mismo tipo. Las activaciones no lineales permiten representar otros patrones. Los datos, el entrenamiento y la evaluación determinan si esa capacidad resulta útil.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-activaciones)


## Tema 3 · 01 · Un ajuste y 09 · Entrenar una red

**Lo que vas a poder hacer:** Ajustar un parámetro y diagnosticar un paso que empeora el resultado.

Ayuda a interpretar una curva de entrenamiento: ajustar más fuerte puede empeorar.

### Entiende la idea

Entrenar es corregir los valores ajustables de una red usando ejemplos con respuestas conocidas. Esos valores, como pesos y sesgos, se llaman parámetros. Primero calculamos una salida y medimos cuánto se aparta de la respuesta; el número que resume ese desacuerdo es la pérdida. Después calculamos cómo cambiaría la pérdida al mover cada parámetro: eso es el gradiente. El tamaño de la corrección depende de una tasa de aprendizaje elegida. En una red, la retropropagación calcula esas influencias a través de sus capas y el optimizador aplica la actualización. Entenderemos el ciclo con un solo ajuste antes de mirar una red completa. La regla de restar tasa por gradiente describe el descenso básico de nuestro ejemplo. Adam, utilizado en el cuaderno, también emplea el historial de los gradientes para ajustar cada parámetro.

### Un caso pequeño antes del programa

**Acercarse a un objetivo sin pasarse**

Partimos de −2 y buscamos 3. La pérdida (parámetro − 3)² empieza en 25. Cada fila comienza otra vez en −2 y realiza un solo paso; no son pasos consecutivos.

| Escenario | Tasa de aprendizaje | Parámetro después | Pérdida |
| --- | --- | --- | --- |
| Inicio | Sin paso | −2 | 25 |
| Pequeño | 0,1 | −1 | 16 |
| Exacto en este ejemplo | 0,5 | 3 | 0 |
| Excesivo | 1,1 | 9 | 36 |

1. En −2, la pendiente de la pérdida es 2 × (−2 − 3) = −10.
2. Con tasa 0,1: parámetro nuevo = −2 − 0,1 × (−10) = −1. La pérdida nueva es (−1 − 3)² = 16.
3. Con tasa 1,1: parámetro nuevo = −2 − 1,1 × (−10) = 9. La pérdida es (9 − 3)² = 36, mayor que las 25 iniciales.

**Qué demuestra:** Desde el mismo inicio, tasa 0,1 reduce la pérdida de 25 a 16 y tasa 1,1 la aumenta a 36. El segundo paso sobrepasa el objetivo.

### Ahora identifica los datos de esta práctica

Primero sigue cómo un paso cambia la pérdida. Después distingue la pérdida de entrenamiento de la comprobación con ejemplos reservados.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Cambia una sola configuración y vuelve a ejecutar el entrenamiento desde su inicio. No confundas una actualización con una época completa.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL1: nombres que encontrarás al calcular y entrenar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DL** — DL: Deep Learning (Aprendizaje profundo). Parte del aprendizaje automático que usa redes neuronales con varias capas para transformar información y aprender representaciones. Una capa transforma números y entrega otros a la siguiente. Más capas no garantizan mejores resultados. [Fuente](https://cloud.google.com/discover/what-is-deep-learning)

- **ReLU** — ReLU: Rectified Linear Unit (Unidad lineal rectificada). Función de activación que devuelve el mayor entre la entrada y cero. Conserva números positivos y convierte los negativos en cero. Su salida no tiene límite superior y no representa por sí sola una probabilidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **BCE** — BCE: Binary Cross Entropy (Entropía cruzada binaria). Pérdida que compara una salida binaria probabilística con su respuesta conocida. En el caso sin pesos calcula −[y ln(p)+(1−y) ln(1−p)]. No tiene unidades físicas y no equivale a un porcentaje de errores; cuanto menor, mejor ajuste de esa pérdida sobre los mismos casos. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

- **BCEWithLogitsLoss** — BCEWithLogitsLoss: Binary Cross Entropy with Logits Loss (Pérdida de entropía cruzada binaria a partir de logits). Función de PyTorch que recibe logits y combina internamente sigmoide con entropía cruzada binaria. Evita separar ambas operaciones para mejorar la estabilidad numérica. En el cuaderno no se añade una sigmoide antes de pasar la salida a esta pérdida. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

- **Adam** — Adam: Adaptive Moment Estimation (Estimación adaptativa de momentos). Optimizador que ajusta los parámetros usando promedios de los gradientes y de sus cuadrados. El nombre procede de esas estimaciones adaptativas. Tiene una tasa global y un ajuste dependiente del historial para cada parámetro; no aplica simplemente el mismo paso del descenso básico. [Fuente](https://arxiv.org/abs/1412.6980)


In [ ]:
# Pérdida L(w) = (w−3)^2. Su derivada es 2(w−3).
TASA = .15 # prueba .02 o 1.1 y compara, no siempre converge
w = -2.; camino = [w]
for paso in range(12):
    w = w - TASA*2*(w-3)
    camino.append(w)
curva=np.linspace(-3,5,120)
plt.plot(curva,(curva-3)**2,color=AZUL);plt.plot(camino,(np.array(camino)-3)**2,'o-',color=VERDE)
plt.title('Descenso del gradiente: pasos hacia menor error');plt.xlabel('Peso w');plt.ylabel('Pérdida (w−3)²');mostrar('01-gradiente')
print('Peso final:',round(w,3),'· error final:',round((w-3)**2,4))


### Después de ejecutar: ¿se acerca o se aleja?

Sigue los puntos desde el inicio. La altura es pérdida, no exactitud ni dinero. Si cambias la tasa, vuelve a ejecutar desde el inicio para comparar ambos recorridos desde −2. Un punto cercano a w=3 es menor pérdida en esta cuenta elegida, no una red validada.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
X,y=datos_cafe(n=320);Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
escalador=StandardScaler().fit(Xtr)
xt=torch.tensor(escalador.transform(Xtr),dtype=torch.float32);yt=torch.tensor(ytr[:,None],dtype=torch.float32)
xe=torch.tensor(escalador.transform(Xte),dtype=torch.float32)
TASA=.02;TAMANO_LOTE=32;EPOCAS=40
torch.manual_seed(SEMILLA)
red=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,8),nn.ReLU(),nn.Linear(8,1))
perdida=nn.BCEWithLogitsLoss() # recibe puntuaciones; no añadir sigmoide antes
opt=torch.optim.Adam(red.parameters(),lr=TASA)
lotes=DataLoader(TensorDataset(xt,yt),batch_size=TAMANO_LOTE,shuffle=True)
historial=[]
for epoca in range(EPOCAS):
    red.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(red(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    historial.append(suma/len(xt))
red.eval()
with torch.no_grad():probs=torch.sigmoid(red(xe)).numpy().ravel()
print('Accuracy en prueba:',accuracy_score(yte,probs>=.5))
fig,axs=plt.subplots(1,2,figsize=(10,4));axs[0].plot(historial,color=AZUL);axs[0].set(title='Aprender: reducir el error',xlabel='Época',ylabel='Pérdida de entrenamiento')
xx,yy=np.meshgrid(np.linspace(0,1,80),np.linspace(0,1,80));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
with torch.no_grad():mapa=torch.sigmoid(red(torch.tensor(escalador.transform(malla),dtype=torch.float32))).numpy().reshape(xx.shape)
axs[1].contourf(xx,yy,mapa,levels=np.linspace(0,1,11),cmap='RdYlGn');axs[1].scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=20);axs[1].set(title='Predecir: usar los pesos aprendidos',xlabel='Uniformidad del color',ylabel='Proporción de manchas');mostrar('09-red-entrenada')


### Después de ejecutar: tres comprobaciones pequeñas

Identifica cuántos ejemplos quedaron fuera del ajuste. Lee la pérdida al comenzar y al terminar sin convertirla a porcentaje. Luego compara el porcentaje de aciertos en esos ejemplos reservados. Si pruebas otra tasa, registra la comparación; elegir con esa reserva la convierte en validación, no en una prueba final nueva.


### DL2: redes e imágenes

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **CNN** — CNN: Convolutional Neural Network (Red neuronal convolucional). Red que usa filtros repetidos sobre pequeñas zonas de una entrada. Los filtros combinan números cercanos y producen mapas de respuestas. La CNN del cuaderno aprende con imágenes de dígitos y sus respuestas conocidas; CNN no significa por sí sola detector de objetos. [Fuente](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


### Lee, comprueba y explica

Primero sigue cómo un paso cambia la pérdida. Después distingue la pérdida de entrenamiento de la comprobación con ejemplos reservados.

**Prueba pequeña:** Cambia una sola configuración y vuelve a ejecutar el entrenamiento desde su inicio. No confundas una actualización con una época completa.

**Vuelve al caso:** Comprueba parámetro anterior, gradiente, tasa, parámetro nuevo y pérdida. Evalúa por separado el funcionamiento con casos reservados.

**Pregunta:** Si baja la pérdida de entrenamiento, ¿ya sabemos que la red generaliza?

**Respuesta razonada:** No. Hay que observar datos separados. Si la pérdida de entrenamiento baja y la de validación empieza a subir, puede haber sobreajuste. La prueba final permanece aparte de las decisiones de desarrollo.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-entrenamiento)


## Cierra el encuentro con una explicación

1. **Una red transforma entradas:** Sigue entradas → productos por pesos → suma con sesgo → activación. Ese recorrido permite comprobar una neurona antes de interpretar una red.
2. **La activación cambia la respuesta:** Compara activaciones manteniendo la entrada fija. Verifica su rango antes de interpretar la salida como puntuación o conectarla a otra operación.
3. **El error guía los ajustes:** Comprueba parámetro anterior, gradiente, tasa, parámetro nuevo y pérdida. Evalúa por separado el funcionamiento con casos reservados.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
